In [2]:
%run ./db_config.ipynb

In [ ]:
price_get_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

price_set_function = {
    "name": "set_ticket_price",
    "description": "Set the ticket price for the traveling city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The name of the city, e.g. London, New York",
            },
            "price": {
                "type": "number",
                "description": "The price of the ticket"
            }
        },
        "required": ["city", "price"],
        "additionalProperties": False
    }
}

book_flight_function = {
    "name": "book_flight",
    "description": "Book a flight for a passenger to a specific destination.",
    "parameters": {
        "type": "object",
        "properties": {
            "passenger_name": {"type": "string", "description": "Full name of the passenger"},
            "destination_city": {"type": "string", "description": "The destination city"}
        },
        "required": ["passenger_name", "destination_city"],
        "additionalProperties": False
    }
}

get_booking_details_function = {
    "name": "get_booking_details",
    "description": "Retrieve flight booking details for a specific passenger.",
    "parameters": {
        "type": "object",
        "properties": {
            "passenger_name": {
                "type": "string", 
                "description": "The full name of the passenger"
            },
        },
        "required": ["passenger_name"],
        "additionalProperties": False
    }
}

In [ ]:
tools = [
    {"type": "function", "function": price_get_function},
    {"type": "function", "function": price_set_function},
    {"type": "function", "function": book_flight_function},
    {"type": "function", "function": get_booking_details_function}
]

In [ ]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"
        

def set_ticket_price(city, price):
    print(f"data saved of city : pirce")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()
        return f"Price {price} for city {city} Saved success fully !!!"
    
def book_flight(passenger_name, destination_city):
    print(f"BOOKING TOOL CALLED for {passenger_name} to {destination_city}")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO bookings (passenger_name, destination, status) VALUES (?, ?, ?)', 
                       (passenger_name, destination_city, "Confirmed"))
        conn.commit()
        return f"Success! Flight to {destination_city} has been booked for {passenger_name}."

def get_booking_details(passenger_name):
    print(f"DATABASE TOOL CALLED: Searching bookings for {passenger_name}")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        # Query the bookings table you created earlier
        cursor.execute('SELECT destination, status FROM bookings WHERE passenger_name = ?', (passenger_name,))
        results = cursor.fetchall()
        
        if not results:
            return f"No bookings found for {passenger_name}."
        
        # Format the results into a readable string
        booking_list = [f"Destination: {row[0]}, Status: {row[1]}" for row in results]
        return f"Bookings for {passenger_name}: " + " | ".join(booking_list)